# Baseline

### Loading the dataset

In [ ]:
import torchvision
from torchvision.datasets import ImageFolder


### Preparing the data
For this baseline, we will take a very simple approach - combine the all labelled data and use it to predict on the test data.

In [ ]:
import os

import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from torchvision.transforms import v2
import pandas as pd

# Define transforms
base_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((224, 224))
])

train_gpu_transform = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomApply([v2.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15, hue=0.02)], p=0.6),
    v2.RandomApply([v2.ElasticTransform(alpha=40.0, sigma=6.0)], p=0.25),
    v2.RandomApply([v2.RandomPerspective(distortion_scale=0.25, p=1.0)], p=0.2),
    v2.RandomApply([v2.RandomAffine(degrees=8, translate=(0.05, 0.05), scale=(0.92, 1.08), shear=4)], p=0.25),
    v2.RandomApply([v2.RandomChoice([
        v2.RandomPhotometricDistort(p=1.0),
        v2.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 1.2))
    ])], p=0.2),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_gpu_transform = v2.Compose([
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class SketchValDataset(Dataset):
    def __init__(self, root_dir, labels_csv, class_to_idx, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        labels_df = pd.read_csv(labels_csv)
        self.samples = []
        for _, row in labels_df.iterrows():
            filename = row['filename']
            class_name = row['class_name']
            if class_name in class_to_idx:
                self.samples.append((filename, class_to_idx[class_name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        filename, label = self.samples[idx]
        img_path = os.path.join(self.root_dir, filename)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label, filename


class SketchTestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = sorted([f for f in os.listdir(root_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        filename = self.image_files[idx]
        img_path = os.path.join(self.root_dir, filename)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, -1, filename


cartoon_ds = ImageFolder('/kaggle/input/competitions/drawn-apart-aicc-round-3/task_data/task_data/cartoon', transform=base_transform)
photo_ds = ImageFolder('/kaggle/input/competitions/drawn-apart-aicc-round-3/task_data/task_data/photograph', transform=base_transform)

# Train with all labeled cartoon + photograph samples
train_ds = ConcatDataset([cartoon_ds, photo_ds])
train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    prefetch_factor=1,
    persistent_workers=True,
)

# Validate only on labeled sketch_val split
val_dataset = SketchValDataset(
    root_dir='/kaggle/input/competitions/drawn-apart-aicc-round-3/task_data/task_data/sketch_val',
    labels_csv='/kaggle/input/competitions/drawn-apart-aicc-round-3/task_data/task_data/val.csv',
    class_to_idx=cartoon_ds.class_to_idx,
    transform=base_transform,
)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=1, persistent_workers=True)

# Inference on sketch_test split
test_dataset = SketchTestDataset(root_dir='/kaggle/input/competitions/drawn-apart-aicc-round-3/task_data/task_data/sketch_test', transform=base_transform)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2, persistent_workers=True)

print(f"Number of training images: {len(train_ds)}")
print(f"Number of images in val dataset: {len(val_dataset)}")
print(f"Number of images in test dataset: {len(test_dataset)}")




### Training the model
We will finetune ResNet34 for this baseline.

In [ ]:
import timm

model = timm.create_model('resnext26ts', pretrained=False)
num_classes = len(cartoon_ds.classes)
model.reset_classifier(num_classes=num_classes)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)



In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import f1_score



In [ ]:
num_epochs = 12
cutmix_alpha = 1.0

best_val_f1 = -1.0
best_model_weights = None


def apply_cutmix(inputs, labels, alpha=1.0):
    if alpha <= 0:
        return inputs, labels, labels, 1.0

    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    batch_size, _, h, w = inputs.shape
    rand_index = torch.randperm(batch_size, device=inputs.device)

    cut_rat = (1.0 - lam) ** 0.5
    cut_w = int(w * cut_rat)
    cut_h = int(h * cut_rat)

    cx = torch.randint(0, w, (1,), device=inputs.device).item()
    cy = torch.randint(0, h, (1,), device=inputs.device).item()

    x1 = max(cx - cut_w // 2, 0)
    x2 = min(cx + cut_w // 2, w)
    y1 = max(cy - cut_h // 2, 0)
    y2 = min(cy + cut_h // 2, h)

    mixed_inputs = inputs.clone()
    mixed_inputs[:, :, y1:y2, x1:x2] = inputs[rand_index, :, y1:y2, x1:x2]

    lam = 1 - ((x2 - x1) * (y2 - y1) / (w * h))
    return mixed_inputs, labels, labels[rand_index], lam


for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
        inputs = train_gpu_transform(inputs.to(device))
        labels = labels.to(device)

        mixed_inputs, labels_a, labels_b, lam = apply_cutmix(inputs, labels, alpha=cutmix_alpha)

        optimizer.zero_grad()
        outputs = model(mixed_inputs)
        loss = lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    # Validation on sketch_val set
    model.eval()
    val_running_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for inputs, labels, _ in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"):
            inputs = val_gpu_transform(inputs.to(device))
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_running_loss += loss.item() * inputs.size(0)

            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_targets.extend(labels.cpu().tolist())

    val_epoch_loss = val_running_loss / len(val_loader.dataset)
    val_f1 = f1_score(all_targets, all_preds, average='weighted')

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {epoch_loss:.4f}, Val Loss: {val_epoch_loss:.4f}, Val F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_weights = model.state_dict()
        print("Saved best model weights!")

print("Model training finished.")



In [ ]:
# Load the best model weights after training
if best_model_weights is not None:
    model.load_state_dict(best_model_weights)
    print(f"Model restored to best validation F1 ({best_val_f1:.4f}) state.")
else:
    print("No best model weights saved.")



# Evaluation

### Loading test dataset

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset

class SketchValDataset(Dataset):
    def __init__(self, root_dir, labels_csv, class_to_idx, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        labels_df = pd.read_csv(labels_csv)
        self.samples = []
        for _, row in labels_df.iterrows():
            filename = row['filename']
            class_name = row['class_name']
            if class_name in class_to_idx:
                self.samples.append((filename, class_to_idx[class_name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        filename, label = self.samples[idx]
        img_path = os.path.join(self.root_dir, filename)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label, filename


class SketchTestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = sorted([f for f in os.listdir(root_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        filename = self.image_files[idx]
        img_path = os.path.join(self.root_dir, filename)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, filename



In [ ]:
val_dataset = SketchValDataset(
    root_dir='/kaggle/input/competitions/drawn-apart-aicc-round-3/task_data/task_data/sketch_val',
    labels_csv='/kaggle/input/competitions/drawn-apart-aicc-round-3/task_data/task_data/val.csv',
    class_to_idx=cartoon_ds.class_to_idx,
    transform=base_transform,
)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=1, persistent_workers=True)

test_dataset = SketchTestDataset(root_dir='/kaggle/input/competitions/drawn-apart-aicc-round-3/task_data/task_data/sketch_test', transform=base_transform)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2, persistent_workers=True)

print(f"Number of images in val dataset: {len(val_dataset)}")
print(f"Number of images in test dataset: {len(test_dataset)}")




### Validation

In [ ]:
model.eval()
all_predictions = []
all_filenames = []

with torch.no_grad():
    for inputs, _, filenames in tqdm(val_loader, desc="Predicting on Validation Data"):
        inputs = val_gpu_transform(inputs.to(device))
        outputs = model(inputs)
        _, predicted_indices = torch.max(outputs, 1)
        all_predictions.extend(predicted_indices.cpu().numpy())
        all_filenames.extend(filenames)

print("Predictions completed.")



In [ ]:
from sklearn.metrics import f1_score

class_names = cartoon_ds.classes
predicted_class_names = [class_names[idx] for idx in all_predictions]
assert len(all_filenames) == len(predicted_class_names), "Mismatch in lengths of filenames and predicted class names!"

val_submission_df = pd.DataFrame({
    'filename': all_filenames,
    'class_name': predicted_class_names
})
val_solution_df = pd.read_csv('val.csv')

merged_df = pd.merge(val_submission_df, val_solution_df, on='filename', suffixes=('_predicted', '_true'))

y_true = merged_df['class_name_true']
y_pred = merged_df['class_name_predicted']

print(f"Validation F1 Score: {f1_score(y_true, y_pred, average='weighted'):.4f}")



### Predict on test

In [ ]:
model.eval()
all_predictions = []
all_filenames = []

with torch.no_grad():
    for inputs, filenames in tqdm(test_loader, desc="Predicting on Test Data"):
        inputs = val_gpu_transform(inputs.to(device))
        outputs = model(inputs)
        _, predicted_indices = torch.max(outputs, 1)
        all_predictions.extend(predicted_indices.cpu().numpy())
        all_filenames.extend(filenames)

print("Predictions completed.")



### Save to CSV

In [ ]:
class_names = cartoon_ds.classes
predicted_class_names = [class_names[idx] for idx in all_predictions]

assert len(all_filenames) == len(predicted_class_names), "Mismatch in lengths of filenames and predicted class names!"

submission_df = pd.DataFrame({
    'filename': all_filenames,
    'class_name': predicted_class_names
})

submission_df.to_csv('submission.csv', index=False)

print("submission.csv created successfully.")
print(submission_df.head())
